# Capitolo 12 — Il Newsvendor e le sue varianti (LP stocastico a scenari)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/laboratorio-ricerca-operativa/blob/main/notebooks/lab12_newsvendor.ipynb)

Caso di studio: panetteria che decide quanti panettoni artigianali produrre.
Prezzo p = 15 €, costo c = 6 €, recupero v = 2 € → Cu = 9, Co = 4.
Domanda normale con media 100 e deviazione standard 20 

Contenuto:
1. Regola del quantile: alpha* = 9/13 = 0,6923 → q* ≈ 110
2. LP a scenari: coincide con il quantile empirico
3. Valore della soluzione stocastica (VSS) e stabilità al numero di scenari
4. Vincoli di servizio (cycle service level, fill rate)
5. Avversione al rischio: frontiera costo medio - CVaR
6. Multiprodotto con budget condiviso e domande correlate

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/laboratorio-ricerca-operativa/newsvendor/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica `stile.py`, la palette comune
degli script del corso. La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: tutti i modelli del laboratorio ci stanno — il più
grande, il newsvendor a scenari, ne usa 1803 e 1801 — ma aumentando il numero di
scenari si può superarla. In quel caso si attiva la licenza accademica gratuita
da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e lo stile grafico del laboratorio.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

if importlib.util.find_spec("stile") is None:
    locale = next((p for p in (Path("../python/stile.py"), Path("python/stile.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))     # notebook aperto nel repository
    else:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/fabiofurini/laboratorio-ricerca-operativa/main/python/stile.py", "stile.py")   # Colab

In [ ]:
import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB
from scipy import stats

from stile import (ARANCIO, GRIGIO, ROSSO, TEAL, VERDE, intestazione, plt, salva_dat,
                   salva_dati, salva_figura)

rng = np.random.default_rng(42)

p, c, v = 15.0, 6.0, 2.0
Cu, Co = p - c, c - v                 # 9 e 4
mu_d, sigma_d = 100.0, 20.0

## 1. REGOLA DEL QUANTILE (soluzione analitica)

In [ ]:
intestazione("Regola del quantile")
alpha_star = Cu / (Cu + Co)
q_star = stats.norm.ppf(alpha_star, mu_d, sigma_d)
print(f"Frattile critico alpha* = Cu/(Cu+Co) = {Cu:.0f}/{Cu + Co:.0f} = {alpha_star:.4f}")
print(f"Quantità ottima q* = F^-1({alpha_star:.4f}) = {q_star:.2f} unità (media = {mu_d:.0f})")


def costo_atteso(q):
    """E[Co(q-D)^+ + Cu(D-q)^+] per domanda normale (funzione di perdita normale)."""
    z = (q - mu_d) / sigma_d
    # E[(D-q)^+] = sigma*(phi(z) - z*(1-Phi(z)))
    perdita = sigma_d * (stats.norm.pdf(z) - z * (1 - stats.norm.cdf(z)))
    ecc = q - mu_d + perdita          # E[(q-D)^+] = q - mu + E[(D-q)^+]
    return Co * ecc + Cu * perdita


print(f"Costo atteso in q*: {costo_atteso(q_star):.2f} €  |  in q = media: "
      f"{costo_atteso(mu_d):.2f} €")

qq = np.linspace(50, 160, 400)
salva_dat(pd.DataFrame({"q": qq, "costo": [costo_atteso(q) for q in qq]}), "cap12_costo")

## 2. LP A SCENARI

In [ ]:
intestazione(f"LP a scenari (S = {600})")
S = 600   # con la licenza pip (2000 var/vincoli) il CVaR richiede S <= 600
dom = np.maximum(rng.normal(mu_d, sigma_d, S), 0.0)
salva_dati(pd.DataFrame({"scenario": range(1, S + 1), "domanda": dom}), "newsvendor_scenari")


def newsvendor_lp(dom_s, prob=None, lam=0.0, alpha_cvar=0.90):
    """LP: min (1-lam)·costo atteso + lam·CVaR_alpha(costo). lam=0 → risk neutral."""
    Sn = len(dom_s)
    pi = np.full(Sn, 1 / Sn) if prob is None else prob
    m = gp.Model("newsvendor")
    m.Params.OutputFlag = 0
    q = m.addVar(name="q")
    o = m.addVars(Sn, name="o")                 # eccedenza
    u = m.addVars(Sn, name="u")                 # carenza
    m.addConstrs((o[s] >= q - dom_s[s] for s in range(Sn)), name="ecc")
    m.addConstrs((u[s] >= dom_s[s] - q for s in range(Sn)), name="car")
    costo_s = {s: Co * o[s] + Cu * u[s] for s in range(Sn)}
    atteso = gp.quicksum(pi[s] * costo_s[s] for s in range(Sn))
    if lam > 0:
        eta = m.addVar(lb=-GRB.INFINITY, name="eta")
        xi = m.addVars(Sn, name="xi")
        m.addConstrs((xi[s] >= costo_s[s] - eta for s in range(Sn)), name="cvar")
        cvar = eta + gp.quicksum(pi[s] * xi[s] for s in range(Sn)) / (1 - alpha_cvar)
        m.setObjective((1 - lam) * atteso + lam * cvar, GRB.MINIMIZE)
    else:
        m.setObjective(atteso, GRB.MINIMIZE)
    m.optimize()
    assert m.Status == GRB.OPTIMAL
    costi = np.array([Co * max(q.X - d, 0) + Cu * max(d - q.X, 0) for d in dom_s])
    return q.X, float(costi @ pi), costi


q_lp, costo_lp, costi_s = newsvendor_lp(dom)
q_emp = np.quantile(dom, alpha_star)
print(f"q ottima dell'LP: {q_lp:.2f}  |  quantile empirico al {alpha_star:.2%}: {q_emp:.2f}")
print(f"Costo atteso (sugli scenari): {costo_lp:.2f} €  |  teorico: {costo_atteso(q_lp):.2f} €")

## 3. VSS E STABILITÀ

In [ ]:
intestazione("Valore della soluzione stocastica (VSS)")
costi_det = np.array([Co * max(mu_d - d, 0) + Cu * max(d - mu_d, 0) for d in dom])
print(f"Decisione 'ingenua' q = E[D] = {mu_d:.0f}: costo atteso {costi_det.mean():.2f} €")
print(f"Decisione stocastica q = {q_lp:.1f}     : costo atteso {costo_lp:.2f} €")
print(f"VSS = {costi_det.mean() - costo_lp:.2f} € per ciclo di vendita")

intestazione("Stabilità: q ottima al variare del numero di scenari")
righe = []
for Sn in [10, 30, 100, 300, 1000, 3000]:
    stime = []
    for rep in range(30):
        dd = np.maximum(rng.normal(mu_d, sigma_d, Sn), 0)
        stime.append(np.quantile(dd, alpha_star))
    righe.append((Sn, np.mean(stime), np.std(stime)))
    print(f"  S = {Sn:5d}: q media {np.mean(stime):7.2f}, dev. std tra repliche {np.std(stime):5.2f}")
stab = pd.DataFrame(righe, columns=["S", "q_media", "q_std"])
salva_dat(stab, "cap12_stabilita")

## 4. VINCOLI DI SERVIZIO

In [ ]:
intestazione("Livelli di servizio")
for beta in [0.90, 0.95, 0.99]:
    q_sl = stats.norm.ppf(beta, mu_d, sigma_d)
    extra = costo_atteso(q_sl) - costo_atteso(q_star)
    print(f"  cycle service level {beta:.0%}: q = {q_sl:6.2f} "
          f"(costo +{extra:5.2f} € rispetto all'ottimo economico)")
q_fill = q_star
fill = 1 - (costo_atteso(q_star) / Cu - Co / Cu * 0) / mu_d  # solo per stampa didattica
perdita_att = sigma_d * (stats.norm.pdf((q_star - mu_d) / sigma_d)
                         - (q_star - mu_d) / sigma_d
                         * (1 - stats.norm.cdf((q_star - mu_d) / sigma_d)))
print(f"  fill rate in q*: {1 - perdita_att / mu_d:.2%} "
      f"(la probabilità di NON avere stock-out è invece {alpha_star:.2%})")

## 5. AVVERSIONE AL RISCHIO: frontiera costo-CVaR

In [ ]:
intestazione("Frontiera costo medio - CVaR (alpha = 0,90)")
alpha_cv = 0.90
front = []
for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    q_l, cm, costi_l = newsvendor_lp(dom, lam=lam, alpha_cvar=alpha_cv)
    var_l = np.quantile(costi_l, alpha_cv)
    cvar_l = costi_l[costi_l >= var_l - 1e-9].mean()
    front.append((lam, q_l, cm, cvar_l))
    print(f"  lambda = {lam:4.2f}: q = {q_l:7.2f}, costo medio = {cm:6.2f}, "
          f"CVaR90 = {cvar_l:6.2f}")
front = pd.DataFrame(front, columns=["lam", "q", "costo_medio", "cvar"])
salva_dat(front, "cap12_frontiera_cvar")
print("Aumentando lambda si ordina di più: costa in media, protegge dagli scenari peggiori.")

## 6. MULTIPRODOTTO CON BUDGET E DOMANDE CORRELATE

In [ ]:
intestazione("Multiprodotto: 3 dolci, budget produzione 1200 €")
nomi_p = ["panettone", "pandoro", "torrone"]
mu_m = np.array([100.0, 80.0, 60.0])
sig_m = np.array([20.0, 25.0, 15.0])
costi_c = np.array([6.0, 5.0, 4.0])
Cu_m = np.array([9.0, 7.0, 5.0])
Co_m = np.array([4.0, 3.5, 2.5])
rho_corr = 0.7
Sigma = np.diag(sig_m) @ (np.full((3, 3), rho_corr) + (1 - rho_corr) * np.eye(3)) @ np.diag(sig_m)
Sm = 300  # limite licenza pip: il multiprodotto ha 3+6S variabili
dom_m = np.maximum(rng.multivariate_normal(mu_m, Sigma, Sm), 0)
budget = 1200.0

mm = gp.Model("newsvendor_multi")
mm.Params.OutputFlag = 0
qm = mm.addVars(3, name="q")
om = mm.addVars(3, Sm, name="o")
um = mm.addVars(3, Sm, name="u")
mm.addConstrs((om[i, s] >= qm[i] - dom_m[s, i] for i in range(3) for s in range(Sm)))
mm.addConstrs((um[i, s] >= dom_m[s, i] - qm[i] for i in range(3) for s in range(Sm)))
v_bud = mm.addConstr(gp.quicksum(costi_c[i] * qm[i] for i in range(3)) <= budget, name="budget")
mm.setObjective(gp.quicksum((Co_m[i] * om[i, s] + Cu_m[i] * um[i, s]) / Sm
                            for i in range(3) for s in range(Sm)), GRB.MINIMIZE)
mm.optimize()
assert mm.Status == GRB.OPTIMAL
print(f"{'prodotto':>10} | {'q senza budget':>14} | {'q con budget':>12}")
for i in range(3):
    q_solo = np.quantile(dom_m[:, i], Cu_m[i] / (Cu_m[i] + Co_m[i]))
    print(f"{nomi_p[i]:>10} | {q_solo:14.1f} | {qm[i].X:12.1f}")
spesa = sum(costi_c[i] * qm[i].X for i in range(3))
print(f"Spesa: {spesa:.2f} / {budget:.0f} €  |  prezzo ombra del budget: {v_bud.Pi:.4f} "
      f"(riduzione del costo atteso per 1 € di budget in più)")

## 7. FIGURE

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.0))
ax1.plot(qq, [costo_atteso(q) for q in qq], color=TEAL, lw=2)
ax1.axvline(mu_d, color=GRIGIO, ls="--", label=f"media = {mu_d:.0f}")
ax1.axvline(q_star, color=ROSSO, ls="-.", label=f"q* = {q_star:.1f}")
ax1.set_xlabel("quantità ordinata q"); ax1.set_ylabel("costo atteso (€)")
ax1.set_title("Il minimo è al 69° percentile, non alla media")
ax1.legend(fontsize=8)
ax2.errorbar(stab["S"], stab["q_media"], yerr=stab["q_std"], fmt="-o", color=TEAL,
             capsize=3)
ax2.axhline(q_star, color=ROSSO, ls="-.", label="q* teorico")
ax2.set_xscale("log")
ax2.set_xlabel("numero di scenari S"); ax2.set_ylabel("q ottima")
ax2.set_title("Stabilità della soluzione a scenari")
ax2.legend(fontsize=8)
salva_figura(fig, "cap12_quantile_stabilita")

fig, ax = plt.subplots()
ax.plot(front["cvar"], front["costo_medio"], "-o", color=TEAL)
for _, r in front.iterrows():
    ax.annotate(f"  $\\lambda$={r['lam']:.2f}, q={r['q']:.0f}", (r["cvar"], r["costo_medio"]),
                fontsize=8)
ax.set_xlabel("CVaR$_{0.90}$ del costo (€)"); ax.set_ylabel("costo medio (€)")
ax.set_title("Frontiera costo medio - rischio di coda")
salva_figura(fig, "cap12_frontiera")

print("\nFatto: capitolo 12.")

---

Notebook generato da `python/lab12_newsvendor.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/laboratorio-ricerca-operativa/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/laboratorio-ricerca-operativa/blob/main/LICENSE-CODE).